# CNN Stock Predictor — demo notebook

A walkthrough of the v3 pipeline: 30-day window of 10 technical
indicators, per-window z-score, predict next-day return, train + eval
around the 2008 crisis.

All logic lives under [`src/`](src/) — this notebook just composes
those pieces. For full experiments across all 10 test tickers (and
v1, v2, v4, v5 variants), see [`experiments/`](experiments/).

In [ ]:
import os
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping

from src import (
    ReturnsCNNConfig,
    build_returns_cnn,
    build_technical_features,
    compute_metrics,
    discover_csv_paths,
    load_csv,
    naive_persistence_forecast,
    prepare_windowed_returns_split,
)

sns.set_style('whitegrid')
plt.style.use('ggplot')

## 1. Load data

Looks for CSVs under `stock_market_data/sp500/csv/` (Kaggle dump).
Pick a ticker that lived through 2008 — JPM is the canonical one.

In [ ]:
csv_paths = discover_csv_paths()
print(f'discovered {len(csv_paths)} tickers')

TICKER = 'JPM'
df = load_csv(csv_paths[TICKER], with_dates=True)
df.tail()

## 2. Build the windowed-returns split with technical features

Train on data up to 2007-07-31, test through 2009-12-31 (the crisis
window). The split function:

- computes 10 indicators (log_return, vol_10/20, momentum_10,
  close_over_sma20, rsi_14, bb_position, volume_z, hl_range_pct,
  co_gap_pct) on the FULL history first
- builds 30-day sliding windows with per-window z-score
- target is the next-day simple return
- internal validation = last 15 % of pre-2007 data (for early stopping)

In [ ]:
split = prepare_windowed_returns_split(
    df,
    train_end='2007-07-31',
    test_start='2007-08-01',
    test_end='2009-12-31',
    window_size=30,
    feature_builder=build_technical_features,
)
print(f'X_train: {split.X_train.shape}')
print(f'X_val:   {split.X_val.shape}')
print(f'X_test:  {split.X_test.shape}')
print(f'y_test return distribution: mean={split.y_test.mean():+.4f} '
      f'std={split.y_test.std():.4f}')

## 3. Build + train the returns CNN

Using the ES-evolved config from v5 (`dropout=0.4`, `huber_delta=0.01`,
`learning_rate=2e-3`) — slightly better than the hand-picked default
on this panel.

In [ ]:
tf.keras.backend.clear_session()
np.random.seed(42); tf.random.set_seed(42)

config = ReturnsCNNConfig(
    dropout=0.4,
    huber_delta=0.01,
    learning_rate=2e-3,
)
model = build_returns_cnn(split.window_size, split.n_features, config=config)
history = model.fit(
    split.X_train, split.y_train,
    validation_data=(split.X_val, split.y_val),
    epochs=60, batch_size=64, verbose=0,
    callbacks=[EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)],
)
print(f'trained {len(history.history["loss"])} epochs (early-stopped)')
print(f'best val_loss = {min(history.history["val_loss"]):.6f}')

## 4. Evaluate against the persistence baseline

The honest comparison: skill score vs `predict tomorrow = today`.

In [ ]:
pred_returns = model.predict(split.X_test, verbose=0).flatten()
pred_prices  = split.close_at_t_test * (1 + pred_returns)
baseline     = naive_persistence_forecast(split.close_at_t_test)

m = compute_metrics(split.actual_close_test, pred_prices, y_prev=split.close_at_t_test)
b = compute_metrics(split.actual_close_test, baseline,    y_prev=split.close_at_t_test)

print(f'{TICKER} on 2008-2009 crisis window:')
print(f'  model:       MAE={m.mae:.3f}  DirAcc={m.directional_accuracy:.1f}%  '
      f'skill_vs_persistence={m.skill_vs_persistence:+.4f}')
print(f'  persistence: MAE={b.mae:.3f}')
print(f'  correlation(pred_returns, actual_returns) = '
      f'{np.corrcoef(pred_returns, split.y_test)[0,1]:+.4f}')

## 5. Plot actual vs predicted vs persistence

Lehman Brothers' bankruptcy (2008-09-15) marked with a dashed line.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(split.test_index, split.actual_close_test, label='actual', color='black', linewidth=1.2)
ax.plot(split.test_index, pred_prices, label='CNN (returns + tech features)', color='C3', alpha=0.85)
ax.plot(split.test_index, baseline, label='persistence', color='C0', alpha=0.4)
ax.axvline(pd.Timestamp('2008-09-15'), color='grey', linestyle='--', alpha=0.7, label='Lehman')
ax.set_title(f'{TICKER} 2007-2009  MAE={m.mae:.2f}  skill={m.skill_vs_persistence:+.4f}')
ax.set_ylabel('Close ($)')
ax.legend()
ax.grid(alpha=0.3)
plt.show()

## What to look at next

- [`experiments/crisis_2008.py`](experiments/crisis_2008.py) — v1 baseline (raw price target, loses badly)
- [`experiments/crisis_2008_v2.py`](experiments/crisis_2008_v2.py) — switch to returns target
- [`experiments/crisis_2008_v3.py`](experiments/crisis_2008_v3.py) — full 10-feature run on all 10 tickers
- [`experiments/multi_ticker_v4.py`](experiments/multi_ticker_v4.py) — one model on the whole S&P 500 (~700 k windows)
- [`experiments/evolve_returns_v5.py`](experiments/evolve_returns_v5.py) — (1+1)-ES on the returns-CNN hyperparameters

See the [README](README.md) for the results table comparing all five.